----
## <font color='CornflowerBlue'>Practical 2: Constructing, Training & Evaluating Neural Networks 
##### Alok Bharadwaj and Arjen Jakobi
---

## Module 1: Fully Connected Networks for Image Classification
<!--      - Build a small FC network in PyTorch
     - Understand train/validation/test splits
     - See overfitting empirically
     - Understand generalization, distribution shift, inductive bias
-->

### Overfitting and Generalisation

In this module, you will build a small fully connected neural network using `PyTorch` and explore how it learns from data. You will work with training, validation, and test splits to understand how model performance is evaluated in practice. Through hands-on experiments, you will observe **overfitting**, where a model learns the training data too well but fails to generalise to new (=unseen) data. This will help you develop an intuition for **generalisation**, the effects of **distribution shift**, and the role of **inductive bias** in shaping how models learn from limited data.

The following code cell sets up the environment by importing the required libraries, loading utility functions, and selecting the available compute device (CPU or GPU) for training.

In [ ]:
# Add src/ to the path so we can import utility functions
import sys
sys.path.append("./src")

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from utils import (
    get_mnist_loaders,
    show_examples,
    plot_loss_curves,
    show_predictions,
    preprocess_drawn_digit,
    predict_drawn_digit,
    show_drawn_digit_prediction,
    make_drawing_canvas,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### 1. The data

<!-- Pointer: introduce MNIST.
     - 60,000 training + 10,000 test, 28x28 grayscale images of digits 0-9
     - We will further split the training set into train + validation
     - Why: train = update parameters; val = monitor overfitting; test = final unseen evaluation
     - Stress: never tune anything based on test performance!
-->

We will use the [MNIST dataset](http://yann.lecun.com/exdb/mnist/), which consists of 28×28 grayscale images of handwritten digits (0–9). The dataset contains 60,000 training images and 10,000 test images.

In this module, we further split the original training set into a **training set** and a **validation set**:
- The **training set** is used to learn/update the model parameters  
- The **validation set** is used to tune/monitor performance and detect overfitting  
- The **test set** is used only for the final evaluation on unseen data  

It is important that the test set remains untouched during model development. No tuning or model selection should be based on test performance.

In the code below, we load these splits using a helper function. You can optionally restrict the size of the training set to intentionally induce overfitting and study its effects. We also visualise a few example images to get an intuition for the data the model will see.

In [ ]:
train_loader, val_loader, test_loader = get_mnist_loaders(
    batch_size=64,
    val_fraction=0.1,
    train_subset_size=None,   # <-- set to e.g. 1000 to deliberately induce overfitting
    seed=0,
)

print(f"Train batches: {len(train_loader)}  ({len(train_loader.dataset)} examples)")
print(f"Val batches:   {len(val_loader)}  ({len(val_loader.dataset)} examples)")
print(f"Test batches:  {len(test_loader)}  ({len(test_loader.dataset)} examples)")

show_examples(train_loader, n=12, title="Some MNIST training examples")

### 2. Build a fully connected network

<!-- Pointer: introduce PyTorch model definition.
     - Subclass nn.Module
     - Define layers in __init__
     - Define the forward pass in forward()
     - Input is [batch, 1, 28, 28]; flatten to [batch, 784] before FC
     - Output is [batch, 10] raw logits
-->

#### 2.1 Defining a model in PyTorch

In this module, we use a **fully connected neural network (FCN)**. In an FCN, each neuron in one layer is connected to every neuron in the next layer. This means the model processes the input as a vector of features, rather than explicitly using the spatial structure of the data.

In PyTorch, neural networks are defined by subclassing `nn.Module`. This provides a flexible way to build models by combining simple building blocks (layers) into more complex architectures.

A model is typically defined in two parts:
- In the `__init__` method, you define the layers of the network  
- In the `forward()` method, you specify how the input data flows through these layers  

The input to the model has shape `[batch, 1, 28, 28]`:
- **batch**: the number of images processed together in one step (for efficiency)  
- **1**: the number of channels (MNIST images are grayscale, so there is only one channel)  
- **28 × 28**: the height and width of each image  

Before passing the data through fully connected (dense) layers, we flatten each image into a vector of shape `[batch, 784]` (each image has 28 × 28 pixels; when we flatten it, we just line up all pixels into a vector, giving 784 input features.). We do this because fully connected (linear) layers expect inputs in the form of **vectors**, not 2D images. Flattening converts the 2D image into a 1D vector so that each pixel becomes an input feature to the network.

---

Input image:
```text
   28 × 28 grid
   ┌─────────────────────────┐
   │ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ▢  ... ▢│
   │ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ▢  ... ▢│
   │ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ▢  ... ▢│
   │         ...             │
   │ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ▢  ... ▢│
   └─────────────────────────┘
```
Flatten row by row:
```text
   [ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ▢ ... ▢ ]
                784 values
```
---

The final layer outputs a tensor of shape `[batch, 10]`, representing the **logits** for the 10 digit classes. 

- Each logit corresponds to one class (digits 0–9)  
- They are not probabilities
- Larger values indicate higher confidence for a class  

To convert logits into probabilities, we apply the **softmax** function. However, in PyTorch, `nn.CrossEntropyLoss` expects **raw logits**, so we do *not* apply softmax in the model. You will learn more about this in section 2c.

### 2.2 Hyperparameters

<!-- Pointer: tell students these are the knobs they will turn to study overfitting.
     Increasing hidden_size or num_layers or epochs while shrinking train_subset_size
     should make overfitting appear.
-->

Before training the model, we need to define a set of **hyperparameters**. These are values that control the structure of the model and the training process, but are not learned from the data.

You can think of hyperparameters as “knobs” that determine how powerful the model is and how it learns. In this module, we will use these hyperparameters to explore overfitting. 

In particular:
- Larger models (more layers or wider layers) can fit the training data more easily  
- More training epochs allow the model to fit the data more closely  
- Less training data makes overfitting more likely  

For the first pass, leave the values as they are. Later on, when evaluating overfitting, try changing these values and observe how the training and validation performance change.

In [ ]:
# ----- Knobs you can turn -----
INPUT_DIM    = 28 * 28   # MNIST images flattened
HIDDEN_SIZE  = 32        # try 32, 128, 1024
NUM_HIDDEN   = 2         # try 1, 2, 4
OUTPUT_DIM   = 10        # 10 digit classes

LEARNING_RATE = 1e-3
NUM_EPOCHS    = 10       # try 5, 20, 50
BATCH_SIZE    = 64       # already used in the loaders above
# --------------------------------

### 2.3 The model class

<!-- Pointer: students fill in the layer definitions and forward pass.
     Show them the class skeleton; the TODOs scope what they have to do.
-->

We now define a fully connected neural network in PyTorch. The model consists of:
- an **input layer** that maps the flattened image to a hidden representation  
- a sequence of **hidden layers** with non-linear activation functions  
- an **output layer** that produces logits for the 10 classes  

Your task is to complete the model definition by specifying the layers and the forward pass. The skeleton below outlines the structure of the network.

---
```python
class FCNet(nn.Module):
    def __init__(self, input_dim, hidden_size, num_hidden, output_dim):
        super().__init__()
        
        # TODO:
        # Define an input layer using nn.Linear
        # It should map from input_dim to hidden_size
        
        # TODO:
        # Define a sequence of hidden layers
        # Each hidden layer consists of:
        #   - nn.Linear(hidden_size, hidden_size)
        #   - nn.ReLU()
        # Repeat this num_hidden times
        # You may find nn.Sequential useful
        
        # TODO:
        # Define an output layer using nn.Linear
        # It should map from hidden_size to output_dim
 
    def forward(self, x):
        # x has shape [batch, 1, 28, 28]
        
        # TODO:
        # Flatten the input to shape [batch, 784]
        # (Hint: use x.flatten(start_dim=1))
        
        # TODO:
        # Pass the input through the input layer
       
        # TODO:
        # Pass the result through the hidden layers
        
        # TODO:
        # Pass through the output layer to obtain logits
        
        return out
```
 
---
 
### Hints
 
- The input is flattened from `[batch, 1, 28, 28]` to `[batch, 784]` 
- `nn.Linear(in_features, out_features)` defines a fully connected layer  
- `nn.ReLU()` applies a non-linear activation  
- `nn.Sequential(...)` allows you to chain layers together  
- The model should output **logits** (do not apply softmax here)  
 
---

In [ ]:
# Insert the class skeleton from above and modify it according to the instructions

### 2.4 Training loop

<!-- Pointer: walk through the standard PyTorch training loop.
     - For each epoch, iterate over batches
     - For each batch: zero gradients, forward, loss, backward, step
     - At end of each epoch, evaluate on validation set with model.eval() and torch.no_grad()
     - Track train/val loss and accuracy each epoch -> plot at the end
-->

We now train the model on the [MNIST]() training set using the standard PyTorch training loop. Training means repeatedly showing the model batches of data, computing how wrong its predictions are, and updating its parameters to reduce this error. Training is done over several **epochs**. In each epoch, the model sees the training data in smaller batches.

The output layer of the network produces one value per class, which as we defined above are called **logits**. These are not probabilities; they can be any real numbers. To interpret them as probabilities, we use the **softmax** function:

$$
p_i = \frac{e^{z_i}}{\sum_{j=1}^{K} e^{z_j}}
$$

where $z_i$ are the logits and $p_i$ are the predicted probabilities. **softmax** has two important properties: (i) all outputs are positive and (ii) the probabilities sum to 1. This allows us to interpret the model’s output as a probability distribution over classes.

---
**Loss function**: For classification tasks, we need a loss function that measures how well the predicted class probabilities match the true labels (here, the labels are the corresponding digits). In this module, we use **cross-entropy loss**, which is standard for multi-class classification. For classification, we want the model to assign high probability to the correct class. Cross-entropy loss measures how different the model’s predicted probability distribution is from the true distribution.

For a single example, the true label can be thought of as a distribution that puts all probability on the correct class. The model, however, outputs a distribution over all classes (via softmax). Cross-entropy quantifies the **mismatch between these two distributions**.

Formally, if $p_i$ are the predicted probabilities and the true label is $y$, the loss is:

$$
\mathcal{L} = -\log(p_y)
$$

Intuitively this means that if the model assigns high probability to the correct class this will result in low loss and if it assigns low probability it will result in high loss, i.e. the model is penalised when it assigns a low probability to the correct class. From an information-theoretic perspective, cross-entropy measures how many bits are needed to encode the true label using the model’s predicted distribution. A better model (closer to the true distribution) leads to lower cross-entropy. For binary classification (i.e. only two classes), a related loss called **binary cross-entropy (BCE)** is often used.

In PyTorch, the function `nn.CrossEntropyLoss` internally applies softmax and computes the negative log-likelihood of the correct class, so we should not apply softmax in the model. The model outputs raw **logits**, one for each class.

---
For each batch, we follow the same sequence:

1. Clear old gradients with `optimizer.zero_grad()`
2. Run the forward pass to compute the logits
3. Compute the loss using cross-entropy loss
4. Run the backward pass with `loss.backward()`
5. Update the parameters with `optimizer.step()`

At the end of each epoch, we evaluate the model on the **validation set** to monitor how well it generalizes to unseen data. This is important because a model can achieve low training loss while still performing poorly on new data (overfitting).

During evaluation, we switch the model to evaluation mode using `model.eval()`. This ensures that certain layers (e.g. dropout, batch normalization) behave consistently and do not introduce randomness. We also use `torch.no_grad()` to disable gradient computation, which reduces memory usage and speeds up evaluation since we are not updating the model.

We store the training and validation loss and accuracy at each epoch so that we can visualise the learning process. These **learning curves** help us diagnose issues such as overfitting (training improves but validation worsens) or underfitting (both remain poor).

In [ ]:
def evaluate(model, loader, criterion, device):
    """Compute average loss and accuracy on a dataloader."""
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n


# Build the model, loss, optimizer
model = FCNet(INPUT_DIM, HIDDEN_SIZE, NUM_HIDDEN, OUTPUT_DIM).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_losses, val_losses, train_accs, val_accs = [], [], [], []

for epoch in range(1, NUM_EPOCHS + 1):
    for x, y in train_loader:
        #     1. move x, y to device
        x = x.to(device)
        y = y.to(device)
        #     2. zero gradients on the optimizer
        optimizer.zero_grad()
        #     3. forward pass
        output = model(x)
        #     4. compute loss
        loss = criterion(output, y)
        #     5. backward pass
        loss.backward()
        #     6. optimizer step
        optimizer.step()
        
    # Evaluate at end of epoch
    train_loss, train_acc = evaluate(model, train_loader, criterion, device)
    val_loss,   val_acc   = evaluate(model, val_loader,   criterion, device)

    train_losses.append(train_loss); val_losses.append(val_loss)
    train_accs.append(train_acc);    val_accs.append(val_acc)

    print(f"Epoch {epoch:>2}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
          f"train_acc={train_acc:.3f}  val_acc={val_acc:.3f}")

### 3. Analysing the training behaviour

<!-- Pointer: this is where the conceptual content lives.
     - Plot loss curves: training loss going to zero while val loss flattens or rises = overfitting
     - Generalization gap = val_loss - train_loss
     - Encourage students to retrain with HIDDEN_SIZE much larger or train_subset_size much smaller and re-plot
     - Compare loss curves before/after
-->

We now visualise the training and validation **loss curves**. These curves provide insight into how well the model is learning and whether it generalises to unseen data.

A typical sign of overfitting is:
- The **training loss** continues to decrease (often approaching zero)  
- The **validation loss** stops improving or starts increasing  

This indicates that the model is memorising the training data rather than learning patterns that generalise. A useful quantity to monitor is the **generalisation gap**, defined as:

$$
\text{gap} = \text{validation\_loss} - \text{training\_loss}
$$

A large gap suggests that the model performs significantly worse on unseen data than on the training data.

---

#### Experiment yourself

To better understand overfitting, try the following for the hyperparameters you have defined in (2a):

- Increase `HIDDEN_SIZE` (make the model larger) 
- Stack number of hiddel layers by increasing `NUM_HIDDEN` 
- Decrease `train_subset_size` (use less training data)  
- Increase `NUM_EPOCHS` (train for more epochs)

Then retrain the model and compare the loss curves. You can also test your model on out-of-distribution data (see 4.)

**Questions to consider:**
- How does the generalisation gap change?  
- Does overfitting become more pronounced?  
- What does this tell you about model capacity and data size?

In [ ]:
plot_loss_curves(train_losses, val_losses, train_accs, val_accs)

### Test set evaluation

<!-- Pointer: this is the *single* evaluation you do once you're done training.
     - Should be done only once per model you settle on
     - If test acc << val acc you may have inadvertently tuned on val
-->

Once you have finished training and are satisfied with your model, you evaluate it on the **test set**. This provides an estimate of how well the model performs on truly unseen data.

The test set should be used only once, after all model design choices and hyperparameters have been fixed. This ensures that it remains an unbiased benchmark of performance.

If the test accuracy is significantly lower than the validation accuracy, it may indicate that the model has been indirectly tuned to the validation set. In that case, the validation performance may be overly optimistic.

In [ ]:
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test loss: {test_loss:.4f}   Test accuracy: {test_acc:.3f}")

# Show some example predictions (mix of correct and wrong)
show_predictions(model, test_loader, device, n=12, only_wrong=False)


### 4. Can the model recognise your handwriting? 

<!-- Pointer: this is the punchline of the module.
     - Even a model that gets 98% test accuracy on MNIST may struggle on hand-drawn digits
     - Why: pencil strokes, centering, image size, contrast all differ from MNIST
     - This is *distribution shift*: the test data comes from a different distribution than train
     - Connect to inductive bias: a model that just learns pixel patterns won't generalize as well
       as a model with appropriate priors (foreshadow CNN module)
-->
As a final experiment, we test the model on handwritten digits that you draw yourself. Even if the model achieves very high accuracy on the MNIST test set (e.g. ~98%), it may struggle with these new inputs. Why?

The reason is that your handwritten digits are not drawn from the same distribution as the MNIST data. Differences in stroke style, thickness, centering, size, and contrast can all affect the model’s predictions. This phenomenon is known as **distribution shift**: the data the model sees at test time differs from the data it was trained on.

This also highlights the role of **inductive bias**. A fully connected network that learns raw pixel patterns may not generalize well to new variations. Models with more appropriate structure (e.g. convolutional neural networks) can better capture the underlying patterns in images and are more robust to such variations. In the next module, we will explore how architectural choices can improve generalisation under distribution shift.

But now, **let us first test this!**



### Draw your own digit

Draw a digit (0–9) using Paint or any drawing tool.

- Save the image as a **PNG file** (any size is fine)  
- Place the file in the same directory as this notebook  
- Update the file path below to point to your image  

In [ ]:
# Edit this path to your saved PNG
image_path = "3.png"

pred, probs = predict_drawn_digit(model, image_path, device)
show_drawn_digit_prediction(image_path, pred, probs)

In [ ]:
# Upload the model, your handwritten digit, loss curves to a shared folder

# upload_to_drive(model)  add this function to the utils